# LIABRARIES

In [ ]:
#!pip install pandas matplotlib numpy seaborn folium geopy panel plotly
#!python -m pip install --upgrade pip
#conda install -c conda-forge param
#!pip install huggingface_hub
#!pip install datasets
#!pip install itables -q

#!pip install --upgrade panel plotly ipywidgets
#!jupyter nbextension enable --py widgetsnbextension
#!jupyter nbextension enable --py panel

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

import folium
from geopy.geocoders import Nominatim


import panel as pn
import plotly.express as px
import plotly.graph_objects as go

import re

import kagglehub
from itables import show

In [ ]:
import plotly.io as pio
pio.renderers.default = "png"  # or "png+notebook" to keep both


### Load Data from Huggings

In [ ]:
path = kagglehub.dataset_download(
    "zillycoco/general-household-dataset-2024"
)

print(path)

In [ ]:
from pathlib import Path

dataset_path = Path(path)

files = list(dataset_path.rglob("*"))

for file in files:
    if file.is_file():
        print(file)

In [ ]:
import pandas as pd

df2 = pd.read_csv("/home/vscode/.cache/kagglehub/datasets/zillycoco/general-household-dataset-2024/versions/1/ghs-2024-person-v1.csv", encoding="cp1252")

# DATA

In [ ]:
df2.head()

In [ ]:

pn.extension('tabulator')

def display_table(df):
    """Displays a DataFrame in a paginated Tabulator widget."""
    table = pn.widgets.Tabulator(df, pagination='remote', page_size=10)
    return pn.Column(table)



In [ ]:


for col in df2.select_dtypes(include='object').columns:
    df2[col] = df2[col].str.replace(r'[–—−]', '-', regex=True)

columns_to_convert = ['uqnr', 'personnr', 'psu']
for col in columns_to_convert:
    if col in df2.columns:
        df2[col] = df2[col].astype(str)


        


In [ ]:
show(df2, paging=True, lengthMenu=[10, 25, 50, 100], maxBytes=0)                    # Display with pagination and search

# Filter 'hhc_fath_alive' == 'Yes' and hhc_fath_parthh' - 'Yes' or 'No'

In [ ]:
filter_df = df2[
    (df2['hhc_fath_alive'] == 'Yes') &
    (df2['hhc_fath_parthh'].isin(['Yes', 'No']))
]

#show(filter_df, paging=True, lengthMenu=[10, 25, 50, 100])
filter_df



In [ ]:

#tab_data.to_excel("tab_data.xlsx", index=False)
# Count the instances of each response in the 'hhc_fath_alive' column
response_counts = filter_df['hhc_fath_parthh'].value_counts()
response_counts


# Data exploration

In [ ]:
# Assuming df2 is your DataFrame
null = df2.isnull().sum().reset_index()
null.columns = ['Column', 'Missing Values']

#show(null, paging=True, lengthMenu=[10, 25, 50, 100])

null

In [ ]:
# Step 1: Get column data types and convert them to string
dtype = df2.dtypes.reset_index()
dtype.columns = ['Column', 'Data type']
dtype['Data type'] = dtype['Data type'].astype(str)  # <--- Convert to str

#show(dtype, paging=True, lengthMenu=[10, 25, 50, 100])
dtype

In [ ]:
summary = df2.describe()
#show(summary, paging=True, lengthMenu=[10, 25, 50, 100])
summary

In [ ]:
## Check for duplicates considering ALL columns
duplicates = df2.duplicated().sum()
print(f"Duplicated rows for df2: {duplicates}")                              #clean_data = clean_data.drop_duplicates()

In [ ]:
df2.shape


# Data transformation (Label encoding - by mapping)  


In [ ]:


for col in df3.select_dtypes(include='object').columns:
    df3[col] = df3[col].str.replace(r'[–—−]', '-', regex=True)

columns_to_convert = ['uqnr', 'personnr', 'psu']
for col in columns_to_convert:
    if col in df3.columns:
        df3[col] = df3[col].astype(str)


In [ ]:

def data_cleaning(df2, df3):
    """
    Cleans df2 using mappings from df3 and returns:
    - cleaned DataFrame
    - list of columns where not all values were affected by the mapping
    - dictionary of unmapped values per partially affected column
    """

    # Make a copy to avoid modifying original
    df2 = df2.copy()

    # 1. Remove 'years' from 'age_grp1' safely
    if 'age_grp1' in df2.columns:
        df2['age_grp1'] = df2['age_grp1'].apply(
            lambda x: x.replace("years", "").strip() if isinstance(x, str) else x
        )

    # 2. Replace 'No fees' with 'None' in fee-related columns
    for col in ['ecd_fees']:
        if col in df2.columns:
            df2[col] = df2[col].replace('No fees', 'None')

    exclude = [
        'uqnr', 'personnr', 'psu', 'stratum', 'person_wgt', 'age',
        'edu_mode_tr', 'edu_totfees', 'lab_salc', 'lab_transp',
        'lab_res_name', 'lab_salary', 'lab_sto'
    ]

    categorical_columns = df2.select_dtypes(include=['object']).columns.tolist()
    clean_df = df2.dropna(subset=categorical_columns).copy()

    partially_affected_columns = []
    unmapped_values = {}

    for col in df3.columns:
        if col in clean_df.columns and col not in exclude:
            original_col = clean_df[col].copy()

            # Create mapping from suffix to full string
            replacements = []
            for _, row in df3.iterrows():
                val = row[col]
                if isinstance(val, str) and '. ' in val:
                    _, suffix = val.split('. ', 1)
                    replacements.append((suffix.strip(), val.strip()))

            mapping_dict = {suffix: full_val for suffix, full_val in replacements}
            unmapped_in_col = set()

            def replace_value(cell):
                cell_str = str(cell).strip()
                if cell_str in mapping_dict:
                    return mapping_dict[cell_str]
                else:
                    unmapped_in_col.add(cell_str)
                    return cell

            clean_df[col] = clean_df[col].apply(replace_value)

            changed = original_col != clean_df[col]
            if not changed.all():
                partially_affected_columns.append(col)
                unmapped_values[col] = list(unmapped_in_col)

    clean_df = clean_df.drop(columns=['spouse', 'spouse_name', 'metro_code'], errors='ignore')

    return clean_df, partially_affected_columns, unmapped_values

# ✅ Example usage
clean_df, not_fully_affected, unmapped_vals = data_cleaning(df2, df3)


In [ ]:
def apply_code_mappings(df):
    def normalize_label(x):
        if isinstance(x, str):
            # Unify en-dash and em-dash to hyphen
            x = re.sub(r'[–—]', '-', x)
            # Replace non-breaking spaces with normal spaces
            x = x.replace('\u00A0', ' ')
            # Remove spaces around dashes for consistency
            x = re.sub(r'\s*-\s*', '-', x)
            # Collapse multiple spaces to single
            x = ' '.join(x.split())
        return x

    df_copy = df.copy()

    mappings = {
        'education': {
            21: "HIGHER/NATIONAL/ADVANCE CERTIFICATE WITH GRADE 12/STD 10/OCCUPATIONAL CERTIFICATE - NQF LEVEL 5",
            10: "GRADE 10/STANDARD 8/FORM 3/NCV LEVEL 2/OCCUPATIONAL CERTIFICATE - NQF LEV EL 2",
            11: "GRADE 11/STANDARD 9/FORM 4/NCV LEVEL 3/OCCUPATIONAL CERTIFICATE - NQF LEV EL 3",
            12: "GRADE 12/STANDARD 10/FORM 5/NATIONAL SENIOR CERTIFICATE/MATRIC/ NCV LEVEL 4 /OCCUPATIONAL CERTIFICATE - NQF LEVEL 4",
            24: "BACHELOR’S DEGREE/OCCUPATIONAL CERTIFICATE - NQF LEVEL 7"
        },
        'ecd_chldatt': {
            2: "Grade R in pre-school that is linked to a primary school"
        },
        'ecd_notatt': {
            5: "Other places than any of the above",
            4: "At someone else’s dwelling"
        },
        'edu_edui': {
            4: "Literacy classes (e.g. Kha Ri Gude)"
        },
        'edu_mode_tr': {
            1: "Walking", 2: "Bicycle", 3: "Motorcycle",
            4: "Minibus taxi/sedan taxi/bakkie taxi",
            5: "Minibus/bus provided by institution/government and not paid for",
            6: "Train", 7: "Bus (public transport)",
            8: "Vehicle hired by a group of parents",
            9: "Own car or other private vehicle",
            10: "Other means of transport",
            88: "Not applicable"
        },
        'edu_time': {
            6: "DO NOT KNOW"
        },
        'edu_totfees': {
            0: "No Fees", 1: "R1–R100", 2: "R101–R200", 3: "R201–R300",
            4: "R301–R500", 5: "R501–R1 000", 6: "R1 001–R2 000", 7: "R2 001–R3 000",
            8: "R3 001–R4 000", 9: "R4 001–R8 000", 10: "R8 001–R12 000",
            11: "R12 001–R16 000", 12: "R16 001–R20 000", 13: "R20 001–R40 000",
            14: "R40 001–R80 000", 15: "More than R80 000", 16: "DO NOT KNOW",
            88: "Not applicable"
        },
        'lab_salc': {
            1: "R1–R2 400", 2: "R2 401–R6 000", 3: "R6 001–R12 000",
            4: "R12 001–R18 000", 5: "R18 001–R30 000", 6: "R30 001–R42 000",
            7: "R42 001–R54 000", 8: "R54 001–R72 000", 9: "R72 001–R96 000",
            10: "R96 001–R132 000", 11: "R132 001–R192 000", 12: "R192 001–R360 000",
            13: "R360 001–R450 000", 14: "R450 001–R650 000", 15: "R650 001–R750 000",
            16: "R750 001–R850 000", 17: "R850 001–R1 000 000", 18: "R1 000 001 or more",
            19: "DO NOT KNOW", 20: "REFUSED", 88: "Not applicable"
        },
        'lab_transp': {
            1: "Office is at home", 2: "Walking", 3: "Bicycle", 4: "Motorcycle",
            5: "Minibus taxi/sedan taxi/bakkie taxi", 6: "Bus", 7: "Train",
            8: "Lift club by a group of people sharing a private vehicle",
            9: "Own car/other private vehicle", 10: "Company car/staff transport",
            11: "Other means of transport", 88: "Not applicable"
        },
        'lab_mins': {
            6: "DO NOT KNOW"
        },
        'geotype': {
            1: "Urban", 2: "Traditional", 3: "Farms"
        },
        'lab_salary': {
            888888888: 'Not applicable', 999999999: 'Unspecified'
        },
        'ecd_care': {
            1: "Biological / adoptive / foster father",
            2: "Biological / adoptive / foster mother",
            3: "Grandfather",
            4: "Grandmother",
            5: "Sister",
            6: "Brother",
            7: "Cousin",
            8: "Aunt",
            9: "Uncle",
            10: "Other relative",
            11: "Paid carer/nanny",
            12: "Someone else",
            13: "NO ONE",
            88: "Not applicable"
        },
        'ecd_birth_cert': {
            1: "Yes",
            2: "No",
            3: "Do not know",
            8: "Not applicable"
        }
    }

    for col, mapping in mappings.items():
        if col in df_copy.columns:
            # Normalize mapping keys
            label_to_prefixed = {
                normalize_label(v): f"{k}. {v}" for k, v in mapping.items()
            }
            # Normalize data values
            df_copy[col] = df_copy[col].apply(normalize_label)
            df_copy[col] = df_copy[col].replace(label_to_prefixed)

    # Special handling for other specific replacements
    if 'edu_grde' in df_copy.columns:
        df_copy['edu_grde'] = df_copy['edu_grde'].replace('14', "14. NC(V) Level 3 (N1/NTC 2)")
    if 'lab_res_name' in df_copy.columns:
        df_copy['lab_res_name'] = df_copy['lab_res_name'].replace('Unspecified', "99. Unspecified")
    if 'lab_sto' in df_copy.columns:
        df_copy['lab_sto'] = df_copy['lab_sto'].replace('Do not Know', "888888888. DO NOT KNOW")

    return df_copy


clean_df = apply_code_mappings(clean_df)

# show(clean_df, paging=True, lengthMenu=[10, 25, 50, 100])
clean_df

# Data Transformation (Cleaning data to remain with codes)

In [ ]:

def data_cleaning(data):
    data = pd.DataFrame(data)

    def keep_numbers(cell):
        cell = str(cell).strip()
        match = re.match(r'^(\d+)\.', cell)
        if match:
            return int(match.group(1))
        elif cell.isdigit():
            return int(cell)
        else:
            return np.nan                   # Explicitly mark as NaN

    for col in data.select_dtypes(include=['object']).columns:
        data[col] = data[col].apply(keep_numbers)

    return data


clean_data = data_cleaning(clean_df)           # Apply your cleaning fuction


for col in clean_data.select_dtypes(include='object').columns:
    clean_data[col] = clean_data[col].str.replace(r'[–—−]', '-', regex=True)

columns_to_convert = ['uqnr', 'personnr', 'psu', 'stratum', 'lab_salary', 'person_wgt', 'lab_sto']
for col in columns_to_convert:
    if col in clean_data.columns:
        clean_data[col] = clean_data[col].astype(str)




#display_table(clean_data)                         #Show table 

# Data Cleaning (filterated data for machine learning )

In [ ]:
# Filter rows where father is alive and part/not part of household (1 or 2)
num_data = clean_data[
    (clean_data['hhc_fath_alive'] == 1) &
    (clean_data['hhc_fath_parthh'].isin([1, 2]))
].copy()  # Use .copy() to avoid SettingWithCopyWarning

# Convert hhc_fath_parthh to binary: 1 -> 1, 2 -> 0
num_data['hhc_fath_parthh'] = num_data['hhc_fath_parthh'].map({1: 1, 2: 0})

# Display the filtered and transformed table
show(num_data, paging=True, lengthMenu=[10, 25, 50, 100])

# Dashboard - Findings

In [ ]:

import requests
import json
import re
import webbrowser
from functools import partial
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
import warnings
warnings.filterwarnings('ignore')


In [ ]:

# Initialize Panel extension
pn.extension('plotly')


In [ ]:

def plot_father_role_by_age_group(df):
    """
    Returns a Panel layout containing a stacked bar chart showing
    father's role in household by age group, using filtered data.
    """

    # Step 2: Ensure columns are strings
    df.loc[:, 'hhc_fath_parthh'] = df['hhc_fath_parthh'].astype(str)
    df.loc[:, 'age_grp1'] = df['age_grp1'].astype(str)

    # Step 3: Create Crosstab and reset index for Plotly
    crosstab = pd.crosstab(
        df['age_grp1'],
        df['hhc_fath_parthh']
    ).reset_index()

    # Step 4: Melt the crosstab to long format
    crosstab_melted = crosstab.melt(
        id_vars='age_grp1',
        var_name='hhc_fath_parthh',
        value_name='Count'
    )

    # Step 5: Plot using Plotly Express
    fig = px.bar(
        crosstab_melted,
        x='age_grp1',
        y='Count',
        color='hhc_fath_parthh',
        title="Distribution of Father Presence at individual level by Age Group<br>(Only where Father is known to be alive)",
        labels={
            'age_grp1': 'Age Group',
            'Count': 'Count',
            'hhc_fath_parthh': "Father Part of HH"
        },
        barmode='stack'
    )

    fig.update_layout(
        xaxis_tickangle=-45,
        legend_title_text='Father Presence',
        margin=dict(t=80, r=20, l=40, b=80)
    )

    # Return as Panel layout
    plotly_pane1 = pn.pane.Plotly(fig)
    return pn.Column(plotly_pane1)


In [ ]:
def plot_province_vs_father_presence(df):
    """
    Returns a Panel layout containing a normalized stacked bar chart 
    of provinces vs father's presence in household,
    excluding 'Not applicable', 'Unspecified', and 'No' values.
    """



    # Step 2: Crosstab (normalized by province)
    crosstab = pd.crosstab(
        df['prov'],
        df['hhc_fath_parthh'],
        normalize='index'
    ).reset_index()

    # Step 3: Melt for Plotly
    df_melted = crosstab.melt(
        id_vars='prov',
        var_name='hhc_fath_parthh',
        value_name='Percentage'
    )

    # Step 4: Plot with Plotly
    fig = px.bar(
        df_melted,
        x='prov',
        y='Percentage',
        color='hhc_fath_parthh',
        barmode='relative',
        title="Distribution of Father Presence at individual level by provinces<br>(Only where father is known to be alive)",
        labels={'prov': 'Province', 'Percentage': 'Proportion'}
    )

    fig.update_layout(
        xaxis_title='Province',
        yaxis_title='Proportion',
        yaxis_tickformat='.0%',
        legend_title="Father's Presence"
    )

    # Return as Panel layout
    return pn.Column(pn.pane.Plotly(fig))


In [ ]:


def plot_correlation_heatmap(df):
    """
    Returns a Panel layout containing a correlation heatmap 
    for all numeric columns in the DataFrame.
    """

    df = pd.DataFrame(filter_df)  # Ensure input is a DataFrame

    # Step 1: Keep only numeric columns
    df_numeric = filter_df.select_dtypes(include=['number'])

    # Step 2: Compute correlation matrix
    df_corr = df_numeric.corr()

    # Step 3: Create Plotly heatmap
    fig = px.imshow(
        df_corr,
        text_auto=True,
        aspect="auto",
        color_continuous_scale="RdBu",
        labels=dict(color="Correlation"),
        title="Correlation Heatmap between Numeric Factors"
    )

    # Step 4: Wrap in Panel layout
    plotly_pane1 = pn.pane.Plotly(fig)
    return pn.Column(plotly_pane1)



In [ ]:

def plot_father_presence_pie(df):
    """
    Returns a Panel layout containing a pie chart showing the distribution 
    of father presence (hhc_fath_parthh), excluding 'Not applicable' responses 
    and where father is not alive, unknown, or unspecified.
    """

    

    # Step 2: Count frequencies
    counts = df['hhc_fath_parthh'].value_counts().reset_index()
    counts.columns = ['hhc_fath_parthh', 'Count']

    # Step 3: Plot pie chart
    fig = px.pie(
        counts,
        names='hhc_fath_parthh',
        values='Count',
        title='Distribution of Father Presence at individual level<br>(Only where father is known to be alive)',
        hole=0.3  # Set to 0.0 for full pie chart 
    )

    fig.update_traces(textinfo='percent+label')

    # Return as Panel layout
    plotly_pane1 = pn.pane.Plotly(fig)
    return pn.Column(plotly_pane1)


In [ ]:

def plot_father_presence_by_race(df):
    """
    Returns a Panel layout containing a grouped bar chart showing 
    father's presence in household across racial groups, 
    excluding 'Not applicable' and where father is not alive, unknown, or unspecified.
    """

   
    # Step 2: Group data
    count_data = df.groupby(['Population', 'hhc_fath_parthh']).size().reset_index(name='Count')

    # Step 3: Plot grouped bar chart
    fig = px.bar(
        count_data,
        x='Population',
        y='Count',
        color='hhc_fath_parthh',
        barmode='group',
        title='Distribution of Father Presence at individual level by<br>Race (Only where father is known to be alive)',
        labels={'Population': 'Race Group', 'hhc_fath_parthh': 'Father Presence'}
    )

    fig.update_layout(
        xaxis_title='Race Group',
        yaxis_title='Count',
        legend_title='Father Presence'
    )

    # Step 4: Return as Panel layout
    plotly_pane1 = pn.pane.Plotly(fig)
    return pn.Column(plotly_pane1)



In [ ]:

def plot_father_presence_by_race(df):
    """
    Returns a Panel layout containing a grouped bar chart showing 
    father's presence in household across racial groups, 
    excluding 'Not applicable' and where father is not alive, unknown, or unspecified.
    """

   
    # Step 2: Group data
    count_data = df.groupby(['Population', 'hhc_fath_parthh']).size().reset_index(name='Count')

    # Step 3: Plot grouped bar chart
    fig = px.bar(
        count_data,
        x='Population',
        y='Count',
        color='hhc_fath_parthh',
        barmode='group',
        title='Distribution of Father Presence at individual level by<br>Race (Only where father is known to be alive)',
        labels={'Population': 'Race Group', 'hhc_fath_parthh': 'Father Presence'}
    )

    fig.update_layout(
        xaxis_title='Race Group',
        yaxis_title='Count',
        legend_title='Father Presence'
    )

    # Step 4: Return as Panel layout
    plotly_pane1 = pn.pane.Plotly(fig)
    return pn.Column(plotly_pane1)

fig = plot_father_presence_by_race(filter_df)
fig

In [ ]:

def plot_distribution(dataframe, x_col='geotype', color_col='hhc_fath_parthh'):
    """
    Plots a stacked bar chart showing the distribution of hhc_fath_parthh within each geotype using Plotly.
    
    Parameters:
    - dataframe (pd.DataFrame): The dataset containing the columns.
    - x_col (str): The categorical column to plot on the x-axis (default 'geotype').
    - color_col (str): The column to use for color grouping and stacking (default 'hhc_fath_parthh').
    """
    fig = px.histogram(
        dataframe,
        x=x_col,
        color=color_col,
        barmode='stack',
        title=f'Stacked Distribution of {color_col} by {x_col}',
        labels={x_col: x_col, color_col: color_col},
    )
    fig.update_layout(
        xaxis_title=x_col,
        yaxis_title='Count',
        bargap=0.1
    )

     # Step 4: Return as Panel layout
    plotly_pane1 = pn.pane.Plotly(fig)
    return pn.Column(plotly_pane1)



In [ ]:
age = plot_father_role_by_age_group(filter_df)
prov =  plot_province_vs_father_presence(filter_df)
heatmap = plot_correlation_heatmap(filter_df)
pie_fath = plot_father_presence_pie(filter_df)
race = plot_father_presence_by_race(filter_df)
geontype = plot_distribution(filter_df) 




In [ ]:

# Define the button first
tableau_button = pn.widgets.Button(name="Open Tableau Dashboard", button_type="primary")

# Optional: add click behavior
def open_tableau(event):
    import webbrowser
    webbrowser.open("https://public.tableau.com/views/Book2_17141594199830/Dashboard2?:language=en-US&:sid=&:redirect=auth&:display_count=n&:origin=viz_share_link")
    
tableau_button.on_click(open_tableau)

# Now include it in your visualizations page
def visualizations_page():
    return pn.Column(
        pn.Row(heatmap.clone(width=500, height=400),
               age.clone(width=600, height=400),
               sizing_mode="stretch_width"),
        pn.Row(prov.clone(width=600, height=400),
               race.clone(width=500, height=400),
               sizing_mode="stretch_width"),
        #pn.Row(pie_fath.clone(width=1100, height=400),
               #sizing_mode="stretch_width"),
        pn.Row(pie_fath.clone(width=500, height=400),
               geontype.clone(width=600, height=400),
               sizing_mode="stretch_width"),
        pn.Row(tableau_button, align='center'),  
        sizing_mode="stretch_width"
    )


In [ ]:

dashboard = visualizations_page()
dashboard.show()


In [ ]:
# Serve the Panel app dashboard.servable()
pn.serve(dashboard)

In [ ]:
SA_GREEN = '#007749'
SA_GOLD = '#FFB81C'
SA_BLUE = '#001489'
SA_RED = '#E4002B'

# Applied to all Plotly charts via color_discrete_map
color_discrete_map={
    'No Father Present': SA_RED,
    'Father Present': SA_GREEN
}

In [ ]:
class StreamingDataHandler:
    def __init__(self, buffer_size=1000):
        self.buffer_size = buffer_size
        self.data_buffer = pd.DataFrame()
        self.update_callbacks = []
    
    def add_data(self, new_batch: pd.DataFrame):
        # Maintains rolling buffer, notifies all dashboard components
        self.data_buffer = pd.concat([self.data_buffer, new_batch])
        if len(self.data_buffer) > self.buffer_size:
            self.data_buffer = self.data_buffer.tail(self.buffer_size)
        self._notify_callbacks()

In [ ]:
# Add to your main app for true streaming:
pn.state.add_periodic_callback(
    lambda: stream_handler.add_data(fetch_new_batch()), 
    period=5000  # Update every 5 seconds
)

# Machine learning 

## Connect Python to R

## Variables for user form

In [ ]:


# --- Group predictors ---
categories = {
    "Education": [
        "education","edu_rsnn","edu_ppriv", "edu_same", "edu_food",  "edu_eatfood", "edu_vlnc"
    ],
    "Labor": [
        "lab_bus", "lab_amount", "lab_sector", "lab_salary"
    ],
    "Social Grants": [
        "soc_grant"
    ],
    "Household": [
        "hhc_moth_alive", "hhc_moth_parthh",
        "hhc_marital", "hhc_relationship", "hlt_preg", "hlt_pregstat"
    ],
    "Demographics": [
        "Sex", "Population", "prov", "geotype", "metro", "disab", "age_grp1"
    ],
    "Employment": [
        "employ_Status2"
    ], 
    "Health": [
        "hlt_medi","hlt_genhealth"
    ]
} 

# --- Categorical code lists ---
categorical_options = { 
    "education": {"0 = GRADE R/0": 0,
        "1. GRADE 1/SUB A/CLASS 1": 1, "2. GRADE 2/SUB B/CLASS 2": 2, "3. GRADE 3/STANDARD 1/ABET/AET 1": 3,
        "4. GRADE 4/STANDARD 2": 4, "5. GRADE 5/STANDARD 3/ABET/AET 2": 5, "6. GRADE 6/STANDARD 4": 6,
        "7. GRADE 7/STANDARD 5/ABET/AET 3": 7, "8. GRADE 8/STANDARD 6/FORM 1": 8,
        "9. GRADE 9/STANDARD 7/FORM 2/ABET/AET 4/NCV LEVEL 1/OCCUPATIONAL CERTIFICATE – NQF LEVEL 1": 9,
        "10. GRADE 10/STANDARD 8/FORM 3/NCV LEVEL 2/OCCUPATIONAL CERTIFICATE – NQF LEVEL 2": 10,
        "11. GRADE 11/STANDARD 9/FORM 4/NCV LEVEL 3/OCCUPATIONAL CERTIFICATE – NQF LEVEL 3": 11,
        "12. GRADE 12/STANDARD 10/FORM 5/NATIONAL SENIOR CERTIFICATE/MATRIC/NCV LEVEL 4/OCCUPATIONAL CERTIFICATE – NQF LEVEL 4": 12,
        "13. NTC L/N1/NQF 1": 13, "14. NTC LL/N2/NQF 2": 14, "15. NTC LLL/N3/NQF 3": 15,
        "16. N4/NTC 4/OCCUPATIONAL CERTIFICATE – NQF LEVEL 5": 16, "17. N5/NTC 5/OCCUPATIONAL CERTIFICATE – NQF LEVEL 5": 17,
        "18. N6/NTC 6/OCCUPATIONAL CERTIFICATE – NQF LEVEL 5": 18, "19. CERTIFICATE WITH LESS THAN GRADE 12/STANDARD 10": 19,
        "20. DIPLOMA WITH LESS THAN GRADE 12/STANDARD 10": 20,
        "21. HIGHER/NATIONAL/ADVANCE CERTIFICATE WITH GRADE 12/STD 10/OCCUPATIONAL CERTIFICATE – NQF LEVEL 5": 21,
        "22. DIPLOMA WITH GRADE 12/STANDARD 10/OCCUPATIONAL CERTIFICATE – NQF LEVEL 6": 22,
        "23. HIGHER DIPLOMA/OCCUPATIONAL CERTIFICATE (B-TECH DIPLOMA) – NQF LEVEL 7": 23,
        "24. BACHELOR’S DEGREE/OCCUPATIONAL CERTIFICATE – NQF LEVEL 7": 24,
        "25. HONOURS DEGREE/POSTGRADUATE DIPLOMA/OCCUPATIONAL CERTIFICATE – NQF LEVEL 8": 25,
        "26. POST HIGHER DIPLOMA (M-TECH AND MASTER'S DEGREE) – NQF LEVEL 9": 26, "27. DOCTORAL DEGREES (D-TECH AND PHD) – NQF LEVEL 10": 27,
        "28. OTHER THAN ANY OF THE ABOVE": 28, "29. DO NOT KNOW": 29, "99. UNSPECIFIED": 99},
    
    "edu_rsnn": {"1. Too old/young": 1, "2. Has completed education/satisfied with my level of education/do not want to study": 2,
        "3. School/education institution is too far": 3, "4. Difficulties to get to school (transport)": 4,
        "5. No money for fees": 5, "6. He or she is working at home or business/job": 6,
        "7. Family commitment (e.g. child minding)": 7, "8. Education is useless or not interesting": 8,
        "9. Unable to perform at school": 9, "10. Illness": 10, "11. Pregnancy": 11, "12. Failed exams": 12,
        "13. Got married": 13, "14. Disability": 14, "15. Violence in school": 15, "16. Not accepted for enrolment": 16,
        "17. Do not have time/too busy": 17, "18. Other reason for not attending an educational institution": 18,
        "88. Not applicable": 88},
    
    "edu_ppriv": {"1. Public (government)": 1,"2. Private (independent)": 2,"3. Do not know": 3,"8. Not applicable": 8}, 
    "edu_food": {"1. YES": 1, "2. NO": 2, "3. DO NOT KNOW": 3, "8. Not applicable": 8},
    "edu_same": {"1. Yes": 1, "2. No": 2, "3. DO NOT KNOW": 3, "8. Not applicable": 8},
    "edu_eatfood": {"1. No": 1, "2. Yes, every day": 2, "3. Yes, a few times a week": 3,
        "4. Yes, sometimes": 4, "5. DO NOT KNOW": 5, "8. Not applicable": 8},
    "edu_vlnc": {"Yes": 1, "No": 2, "DO NOT KNOW": 3, "Not applicable": 8},
    "lab_bus": {"1. Yes": 1, "2. No": 2, "8. Not applicable": 8, "9. Unspecified": 9}, 
    "lab_sector": {"1. In the formal sector (registered to perform an activity)": 1,
        "2. In the informal sector (not registered to perform an activity)": 2,
        "3. DO NOT KNOW": 3, "8. Not applicable": 8}, 
    "lab_salary": {"Valid range": "1–1,400,000", "Not applicable": 888888888, "Unspecified": 999999999},
    "lab_amount": {"1. Yes": 1, "2. No": 2, "3. DO NOT KNOW": 3, "8. Not applicable": 8},
    "soc_grant": {"1. Yes": 1, "2. No": 2, "3. Do not know": 3},

    # Demographics
    "prov": {"1. Western Cape": 1, "2. Eastern Cape": 2, "3. Northern Cape": 3, "4. Free State": 4,
             "5. KwaZulu-Natal": 5, "6. North West": 6, "7. Gauteng": 7, "8. Mpumalanga": 8, "9. Limpopo": 9},
    "Sex": {"1. Male": 1, "2. Female": 2}, 
    "age_grp1": {"1. 00–04 years": 1, "2. 05–09 years": 2, "3. 10–14 years": 3, "4. 15–19 years": 4,
                 "5. 20–24 years": 5, "6. 25–29 years": 6, "7. 30–34 years": 7, "8. 35–39 years": 8,
                 "9. 40–44 years": 9, "10. 45–49 years": 10, "11. 50–54 years": 11, "12. 55–59 years": 12,
                 "13. 60–64 years": 13, "14. 65–69 years": 14, "15. 70–74 years": 15, "16. 75+": 16},
    "Population": {"1. African/Black": 1, "2. Coloured": 2, "3. Indian/Asian": 3, "4. White": 4},
    "geotype": {"1. Urban": 1, "2. Traditional": 2, "3. Farms": 3}, 
    "metro": {
        "1. Metro": 1, "2. Non-metro": 2},
    "disab": {"0. Not disabled": 0, "1. Disabled": 1},
    "employ_Status2": {"1. Employed": 1, "2. Unemployed": 2, "3. Not economically active": 3, "8. Not applicable": 8},
    "hlt_medi": {"1. Yes": 1, "2. No": 2, "3. DO NOT KNOW": 3},
    "hlt_genhealth": { "1. Excellent": 1, "2. Very good": 2, "3. Good": 3, "4. Fair": 4, "5. Poor": 5, "6. NOT SURE": 6},
    "hhc_marital": {"1. Legally married": 1, "2. Living together like husband and wife/partners": 2, "3. Divorced": 3, 
                    "4. Separated, but still legally married": 4, "5. Widowed": 5,"6. Single, but have lived together with someone as husband/wife before": 6, 
                    "7. Single and have never been married/never lived together as husband/wife before": 7, "9. Unspecified":  9},
    "hhc_moth_alive": {"1. Yes": 1, "2. No": 2, "3. Do not know": 3, "9. Unspecified": 9},
    "hhc_moth_parthh": {"1. Yes": 1, "2. No": 2, "8. Not applicable": 8},
    "hhc_relationship": {"1. Head/acting head": 1, "2. Husband/wife/partner of household head": 2,"3. Son/daughter/stepchild/adopted child of household head": 3, 
                   "4. Brother/sister/stepbrother/stepsister of household head": 4, "5. Father/mother/stepfather/stepmother of household head": 5,
                   "6. Grandparent/great grandparent of household head": 6,"7. Grandchild/great grandchild of household head": 7,
                   "8. Other relative (e.g. in-laws or aunt/uncle) of household head": 8,"9. Non-related persons": 9,"99. Unspecified": 99},
    "hlt_preg": { "1. Yes": 1, "2. No": 2, "3. DO NOT KNOW": 3, "8. Not applicable": 8}, 
    "hlt_pregstat": {"1. Currently still pregnant": 1, "2. The child has been born alive": 2, 
        "3. The child died in the womb or during childbirth after the 7th month of pregnancy (stillbirth)": 3,
        "4. The child died in the womb or the pregnancy ended before the 7th month of pregnancy (spontaneous abortion/miscarriage)": 4,
        "5. The pregnancy was ended by choice before the child was born (termination of pregnancy/abortion by choice)": 5, "8. Not applicable": 8}

}


# --- Help text ---
categorical_help = {
    "prov": "What province are you located?",
    "Sex": "What is your biological gender?",
    "age_grp1": "What is your age group?",
    "Population": "What is your race? ",
    "geotype": "Choose the geography type based on your area.",
    "metro": "Is your area metro or non-metro.",
    "disab": "What is your disability status?",
    "employ_Status2": "What is your employment status?", 
    "hlt_medi": "Are you covered by a medical aid or medical benefit scheme or other private health insurance?",
    "hlt_genhealth": "How is your general health?",
    

    "hhc_relationship": "What is your relationship with the head of the household?",
    "hhc_marital": "What is your marital status?",
    "hhc_moth_alive": "Is your biological mother alive?",
    "hhc_moth_parthh": "Based on the previous question, is your biological mother part of the household?",
    "hlt_preg": "Have you been pregnant during the past 12 months?", 
    "hlt_pregstat": "Based on the previous question, what is the current status of this pregnancy?",

    
    "education": "What is your highest level of education that you have successfully completed?",
    "edu_food": "Do you get food at school as part of the school feeding scheme/government nutrition programme?",
    "edu_eatfood": "Based on the previous questio, how regularly do you collect the food provided by school feeding scheme?",
    "edu_same": "How you doing the same grade as the preceding year?",
    "edu_rsnn": "What is the main reason why you are not currently attending any educational institution?",
    "edu_ppriv": "Is the institution that you are attending public or private?",
    "edu_vlnc": "Have you experienced any form of violence, corporal punishment or verbal abuse at school over the past 3 months (eg. bulling, teased, harassed or insulted)?", 
    "lab_bus": "In the last week (Mon-Sun), did you run or do any kind of business (big or small) for yourself or with one or more partners, even if it was only for one hour?\n (Eg: commercial farming, selling or making things for sale, construction, repairing things, guarding cars, brewing beer, collecting wood or water for sale, hairdressing, crèche businesses, taxi or other transport businesses, legal or medical practice, performing in public, operating a public phone shop, etc.)",
    "lab_amount": "Willing to disclose earnings?",
    "lab_sector": "What is the sector of the organisation/business/branch where you work?",
    "lab_salary": "What is your total monthly salary including overtime, allowances, bonuses and any deductions before tax?",
    
    "soc_grant": "Do you take social grant?",
    
}


## User interface website layout

In [ ]:
#!pip install requests panel


In [ ]:
import panel as pn
import requests
from functools import partial
import json  # For proper JSON display

pn.extension()

API_URL = "http://127.0.0.1:47260"  # Your Plumber API URL

# -----------------------------
# Helper: widget creation
# -----------------------------
def make_input_widget(col, categorical_options, categorical_help, selected_values):
    if col in categorical_options:
        select_widget = pn.widgets.Select(
            options=categorical_options[col],
            value=list(categorical_options[col].values())[0],
            width=250
        )
        selected_values[col] = select_widget.value

        select_widget.param.watch(
            partial(lambda name, event, sv: sv.__setitem__(name, event.new), col, sv=selected_values),
            "value"
        )

        label = categorical_help.get(col, col)

        toggle = pn.widgets.Toggle(
            name=f"▶ {label}",
            button_type="default",
            width=300 
        )
        toggle.param.watch(
            partial(lambda tgl, event, name: setattr(
                tgl, "name",
                f"▼ {categorical_help.get(name, name)}" if event.new else f"▶ {categorical_help.get(name, name)}"
            ), toggle, name=col),
            "value"
        )

        toggle_row = pn.Row(
            pn.pane.Markdown(f"<span title='Click to select {label.lower()}'></span>", width=1),
            toggle,
        )

        return pn.Column(
            toggle_row,
            pn.bind(lambda show, w=select_widget: w if show else pn.pane.Markdown(""), toggle),
            sizing_mode="stretch_width"
        )

    else:
        num_widget = pn.widgets.FloatInput(name=col, value=0)
        selected_values[col] = num_widget.value
        num_widget.param.watch(
            partial(lambda name, event, sv: sv.__setitem__(name, event.new), col, sv=selected_values),
            "value"
        )
        return num_widget

# -----------------------------
# Build widgets & accordion
# -----------------------------
inputs = {}
selected_values = {}
for group, cols in categories.items():
    for col in cols:
        inputs[col] = make_input_widget(col, categorical_options, categorical_help, selected_values)

accordion_items = [(group, pn.Column(*(inputs[col] for col in cols))) for group, cols in categories.items()]
accordion = pn.Accordion(*accordion_items, active=[0], sizing_mode="stretch_width")

accordion_container = pn.Column(accordion, sizing_mode="stretch_both", scroll=True)

# -----------------------------
# Prediction section
# -----------------------------
pred_output = pn.pane.Markdown("### Prediction will appear here.")
predict_button = pn.widgets.Button(name="Predict", button_type="primary")

def predict_callback(event):
    user_input = {col: selected_values[col] if col in selected_values else inputs[col].value
                  for col in inputs.keys()}

    # Show user input JSON
    pred_output.object = f"### 📝 User Input JSON\n```json\n{json.dumps(user_input, indent=2)}\n```"

    try:
        response = requests.post(f"{API_URL}/predict", json=user_input)
        if response.status_code == 200:
            result = response.json()
            pred_class = result.get("prediction", "Unknown")

            # Convert prediction list to scalar if needed
            if isinstance(pred_class, list):
                pred_class = pred_class[0]

            prob_yes = result.get("probability_yes", 0.0)
            prob_no  = result.get("probability_no", 0.0)
            if isinstance(prob_yes, list):
                prob_yes = float(prob_yes[0])
            if isinstance(prob_no, list):
                prob_no = float(prob_no[0])

            pred_output.object += (
                f"\n\n### 🔮 Prediction: **{pred_class}**\n"
                f"- Prob(Yes) = {prob_yes:.2f}\n"
                f"- Prob(No)  = {prob_no:.2f}"
            )

            global last_user_input, last_prediction
            last_user_input = user_input
            last_prediction = pred_class
        else:
            pred_output.object += f"\n\n### ❌ API Error: {response.text}"
    except Exception as e:
        pred_output.object += f"\n\n### ❌ Connection Error: {e}"

predict_button.on_click(predict_callback)

# -----------------------------
# Feedback
# -----------------------------
feedback_radio = pn.widgets.RadioButtonGroup(
    name='Feedback',
    options={'Yes (Correct)': 1, 'No (Incorrect)': 0},
    button_type='success'
)
feedback_button = pn.widgets.Button(name="Submit Feedback", button_type="success")
feedback_output = pn.pane.Markdown("### Feedback not submitted yet.")

def submit_feedback(event):
    if "last_user_input" not in globals():
        feedback_output.object = "### ❌ Make a prediction first."
        return
    user_feedback = feedback_radio.value
    if user_feedback is None:
        feedback_output.object = "### ⚠️ Please select feedback before submitting."
        return
    new_row = last_user_input.copy()
    new_row["hhc_fath_parthh"] = int(user_feedback)
    try:
        response = requests.post(f"{API_URL}/feedback", json=new_row)
        if response.status_code == 200:
            result = response.json()
            feedback_output.object = (
                f"### ✅ Feedback submitted.\n\n"
                f"Training rows: {result.get('rows', 'N/A')}"
            )
            feedback_radio.value = None
        else:
            feedback_output.object = f"### ❌ API Error: {response.text}"
    except Exception as e:
        feedback_output.object = f"### ❌ Connection Error: {e}"

feedback_button.on_click(submit_feedback)

feedback_section = pn.Column(
    pn.pane.Markdown("## 📝 Provide Feedback"),
    feedback_radio, feedback_button, feedback_output,
    sizing_mode="stretch_width"
)

# -----------------------------
# Retrain (FIXED)
# -----------------------------
retrain_button = pn.widgets.Button(name="Retrain Model", button_type="warning")
retrain_output = pn.pane.Markdown("")

def retrain_callback(event):
    try:
        response = requests.post(f"{API_URL}/retrain", json={})
        if response.status_code == 200:
            result = response.json()

            # Fix: handle oob_error if it's a list
            oob_error = result.get("oob_error", 0)
            if isinstance(oob_error, list) and len(oob_error) > 0:
                oob_error = float(oob_error[0])
            else:
                oob_error = float(oob_error) 

            retrain_output.object = (
                f"### 🔄 Retrained Model\n\n"
                f"- Status: {result.get('status', 'N/A')}\n"
                f"- OOB Error: {oob_error:.3f}"
            )
        else:
            retrain_output.object = f"### ❌ API Error: {response.text}"
    except Exception as e:
        retrain_output.object = f"### ❌ Connection Error: {e}"

retrain_button.on_click(retrain_callback)

# -----------------------------
# Layout containers
# -----------------------------
buttons_container = pn.Column(
    pn.Row(predict_button, sizing_mode="stretch_width"),
    pn.Row(pred_output, sizing_mode="stretch_width"),
    feedback_section,
    pn.Row(retrain_button, sizing_mode="stretch_width"),
    pn.Row(retrain_output, sizing_mode="stretch_width"),
    sizing_mode="stretch_width"
)

prediction_page = pn.Column(
    pn.pane.Markdown("# 👨‍👩‍👧 Household Father Presence Prediction"),
    "Expand categories to enter values, then click **Predict**.",
    accordion_container,
    buttons_container,
    sizing_mode="stretch_both"
)

# -----------------------------
# Other pages
# ----------------------------- 
cover = pn.pane.PNG("Presentation1.png", sizing_mode="stretch_both", height=800)
home_page = pn.Column("# 🏠 Home", cover, "Welcome to the Father Presence Prediction App!", sizing_mode="stretch_both")
dashboard_page = pn.Column("# 📊 Dashboard", "📈 Dashboard content placeholder", sizing_mode="stretch_both")
findings_page = pn.Column(
    "# 🔍 Findings",
    """
## **General Report on Father Presence Analysis in South African Households**

### **1. Introduction**

The interactive dashboard visualizes the underlying trends and relationships within the dataset used to predict **father absence in South African households**. Its purpose is to uncover how **demographic, geographic, and socio-economic variables** influence the likelihood of a father figure being present. Through statistical modeling and visual exploration, this analysis provides valuable insights into family structure patterns and their underlying socio-economic drivers.

---

### **2. Data Insights and Correlation Analysis**

A **correlation heatmap** was employed to examine the relationships among numeric variables such as **age, stratum, and person weight**. The results showed **weak correlations**, indicating that **multicollinearity is minimal**.  
This enhances **model reliability** and ensures that each variable contributes independently to predicting father absence, improving both accuracy and interpretability.

---

### **3. Demographic Distribution**

The **distribution by age group** reveals that **children aged 0–14 years** are the most affected by father absence. The likelihood of father presence **declines progressively with age**, indicating that younger children are more likely to grow up without their fathers compared to older individuals.  
This pattern may reflect factors such as **migration for employment**, **relationship dissolution**, or **economic instability**—all of which disproportionately affect younger households.

---

### **4. Geographic and Socio-Economic Patterns**

Provincial-level analysis highlights regional disparities:

* **Eastern Cape, KwaZulu-Natal, and Limpopo** exhibit the **highest father absence rates**, potentially due to economic hardship and labor migration.
* The **Western Cape** records a **lower rate** of father absence, suggesting stronger family cohesion and more stable socio-economic conditions.

The **race-based distribution** illustrates disparities consistent with **broader socio-economic inequalities**, reinforcing the interconnectedness of race, income, and family structure.

---

### **5. Geotype (Settlement Type) Analysis**

When disaggregated by **geotype** (rural, urban, and traditional):

* **Urban areas** demonstrate **greater variability** in father presence, reflecting diverse socio-economic contexts.
* **Traditional households** show **higher father absence rates**, likely influenced by cultural, economic, or migratory factors.

This spatial differentiation underlines the importance of contextual understanding when designing social interventions aimed at promoting family stability.

---

### **6. Overall Distribution**

The **overall distribution pie chart** indicates that **69% of individuals live without a father figure**, while **31% live with one**.  
This finding aligns with **national demographic statistics**, confirming the dataset’s validity. It also underscores the **urgency of addressing fatherlessness** as a significant social issue with implications for child development, education, and long-term well-being.

---

### **7. Tableau Bubble Chart Visualizations**

This report also analyzes the **distribution of father presence** using **bubble charts from Tableau**, exploring how it relates to **mother presence**, **social grant dependency**, and **age group**.

#### **7.1 Distribution of Father Presence by Mother Presence**

The first bubble chart, *“Distribution of Father Presence by Mother Presence (Only Where Father Is Known to Be Alive)”*, compares the co-presence of mothers and fathers in households.

* **Blue bubbles** represent instances where the father is **not present**, while **red bubbles** show where the father **is present**.
* The chart indicates that households where fathers are **absent** are more frequent than those where fathers are present, even when the father is known to be alive.

This pattern suggests potential **social or economic factors** influencing family composition, possibly linked to **migration**, **employment location**, or **relationship separation**.

#### **7.2 Relationship Between Father Presence and Social Grant Dependency Across Age Groups**

The second chart examines how **father presence** correlates with **social grant dependency** across **different age groups**.

* A concentration of **blue bubbles** indicates that father absence is more common among **children in households dependent on social grants**.
* The smaller **red clusters** imply fewer cases where fathers are present in grant-receiving households.

This suggests a relationship between **father absence and financial vulnerability**, as households lacking a resident father may rely more heavily on government assistance to support children’s needs.

#### **7.3 Father Presence by Household Relationship and Age Group**

The third visualization, *“Father Presence by Household Relationship and Age Group,”* integrates **household structure and age**.

* **Bubble sizes** correspond to the **number of children** within each category.
* In the **05–09-year age group**, the tooltip shows **3,310 children** from households where fathers are **not present** and who receive **social grants**.
* The **larger bubbles** represent **grandchildren or great-grandchildren**, indicating that children in **extended family settings** are the most affected by father absence.

This pattern shows that **younger age groups (particularly under 10 years)** exhibit **higher instances of father absence** combined with **social grant dependency**.  
As children grow older, the proportion of “Yes” (father present) bubbles increases slightly, though **“No” remains dominant**.


---

### **8. Feature Importance Analysis**

A **feature importance ranking** (derived from a Random Forest model) identified the most influential variables associated with predicting father presence.

| **Variable**                                  | **Mean Decrease Accuracy** | **Mean Decrease Gini** | **Interpretation**                                                                                |
| --------------------------------------------- | -------------------------- | ---------------------- | ------------------------------------------------------------------------------------------------- |
| **hhc_relationship**                          | 83.14                      | 2207.22                | Household relationship is the most influential predictor, emphasizing family role structure.      |
| **hhc_moth_parthh**                           | 47.11                      | 713.29                 | Mother’s presence in the household strongly predicts father presence.                             |
| **Population**                                | 41.32                      | 189.80                 | Larger populations correlate with patterns of father absence due to urbanization effects.         |
| **soc_grant**                                 | 33.13                      | 198.21                 | Social grant dependency is closely linked to father absence, reflecting economic hardship.        |
| **education / edu_food / edu_eatfood**        | 20–27                      | 50–80                  | Educational and food security variables play moderate roles, reflecting socio-economic influence. |
| **prov**                                      | 29.56                      | 235.88                 | Province of residence is a strong geographical determinant of father presence.                    |
| **age_grp1**                                  | 29.42                      | 257.12                 | Age group confirms that children and youth are most affected by father absence.                   |
| **geotype / metro**                           | 26–28                      | 98–126                 | Settlement type and metropolitan status influence family dynamics and stability.                  |
| **hhc_marital**                               | 23.75                      | 271.21                 | Marital status of household members remains an important family structure indicator.              |
| **hlt_medi / employ_Status2 / lab variables** | 15–27                      | 55–126                 | Employment and health-related variables contribute moderately to father presence likelihood.      |

In summary:

* **Household structure variables** (e.g., *relationship*, *mother’s presence*) are the **strongest predictors**.  
* **Socio-economic factors** (e.g., *social grants*, *education*, *employment*) significantly influence father presence.  
* **Geographic context** (*province*, *geotype*) adds spatial variation to family composition trends.
* **Age factor:** **Younger children** are more likely to live without a father present, often in **extended or multigenerational households**.

---

### **9. Conclusion and Policy Implications**

This comprehensive analysis validates the dataset’s structure and highlights critical variables influencing father presence.  
The findings emphasize the **multi-dimensional nature of father absence**, driven by a blend of demographic, socio-economic, and geographic factors.

Overall, this analysis reinforces that **father absence in South Africa is both a social and economic issue**, requiring coordinated responses across welfare, education, and labor sectors.
    """,
    sizing_mode="stretch_both"
)


recommendations_page = pn.Column(
    "# 📝 Recommendations",
    """


    To address these challenges, policymakers and social development organizations should consider:

    * **Targeted family support initiatives** promoting father engagement.  
    * **Socio-economic interventions** reducing financial pressures that contribute to father absence.  
    * **Education and employment programs** aimed at stabilizing households.  
    * **Regional and cultural awareness** when implementing family policy reforms.
    """,
    sizing_mode="stretch_both"
)


content_dict = {
    "Home": home_page,
    "Prediction": prediction_page,
    "Dashboard": dashboard,
    "Findings": findings_page,
    "Recommendations": recommendations_page
}

nav_bar = pn.Row(
    pn.pane.PNG("logo.png", width=60, height=60, sizing_mode="fixed"),
    pn.pane.Markdown("## Father Presence Prediction App", styles={"color": "white"}),
    pn.layout.HSpacer(),
    *[pn.widgets.Button(name=key, button_type="primary") for key in content_dict.keys()],
    sizing_mode="stretch_width",
    styles={"background": "black", "position": "sticky", "top": "0px", "z-index": "1000", "padding": "10px"}
)

main_panel = pn.Column(content_dict["Home"], sizing_mode="stretch_both")

def switch_page(event):
    page = event.obj.name
    main_panel.clear()
    main_panel.append(content_dict[page])

for w in nav_bar:
    if isinstance(w, pn.widgets.Button):
        w.on_click(switch_page) 

# -----------------------------
# Final App
# -----------------------------
app = pn.Column(nav_bar, main_panel, sizing_mode="stretch_both")
app.show()


In [ ]:
# CELL 1: Setup
import pandas as pd
import numpy as np
import panel as pn
import plotly.express as px

pn.extension('plotly')

# CELL 2: Data
filter_df = df2[
    (df2['hhc_fath_alive'] == 'Yes') &
    (df2['hhc_fath_parthh'].isin(['Yes', 'No']))
].copy()

# CELL 3: Correlation heatmap
df_numeric = filter_df.select_dtypes(include=['number'])
df_corr = df_numeric.corr()

fig1 = px.imshow(
    df_corr,
    text_auto='.2f',
    aspect="auto",
    color_continuous_scale="RdBu_r",
    zmin=-1, zmax=1,
    title="Correlation Heatmap"
)
fig1.update_layout(width=500, height=400)

# CELL 4: Bar chart
crosstab = pd.crosstab(filter_df['age_grp1'], filter_df['hhc_fath_parthh']).reset_index()
crosstab_melted = crosstab.melt(id_vars='age_grp1', var_name='Father_Presence', value_name='Count')

fig2 = px.bar(
    crosstab_melted,
    x='age_grp1', y='Count', color='Father_Presence',
    title="Father Presence by Age Group",
    barmode='stack'
)
fig2.update_layout(xaxis_tickangle=-45, width=600, height=400)

# CELL 5: Dashboard (run this LAST in its own cell)
dashboard = pn.Column(
    pn.Row(pn.pane.Plotly(fig1), pn.pane.Plotly(fig2))
)
dashboard.show()

In [ ]:
#!jupyter nbconvert --to pdf Part1_Pythoncode.ipynb